# USE THIS WITH THE LABELED DATA!

In [ ]:
WINDOW_SIZE = 4
IS_TRAINED = True

In [ ]:
import pyarrow.dataset as ds
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
import pickle

file_path = '../data/sub/train_data.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()

### Encode all columns so that they are numerical

In [ ]:
label_encoders = {}
for col in df.select_dtypes(include='object').columns:  # 'object' dtype selects categorical columns
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

### Separate X from y

In [ ]:
# Dynamically construct the FLOW column names
flow_columns = [f"FLOW_{i}" for i in range(1, WINDOW_SIZE + 1)]

# Define the full feature list
base_features = ['USAGE', 'HOUSING', 'WEEKDAY', 'START_HOUR']
features = base_features + flow_columns
target = 'LEAK'

# Select features and target
X = df[features]
y = df[target]

### Separate Train data, Validation data, Test data

TODO: Import the separate_data.ipynb to be usable here

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Model fitting, Prediction

In [ ]:
if (IS_TRAINED):
    model = pickle.load(open("../data/models/test_model.sav", 'rb'))
else:
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    model.fit(X_train, y_train)
    pickle.dump(model, open("../data/model_test.sav", 'wb'))

In [ ]:
y_pred = model.predict(X_test)

### Results

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))